In [1]:
import cv2
import json
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import sys
from pathlib import Path
import h5py
from scipy.io import loadmat

# Add src to path
sys.path.append(str(Path.cwd().parent.parent / "src"))
from monkey_fetus_utils import assign_coda_classifications_to_geojson, polygon_centroid


# ============================================================
# GS40 CODA Integration Configuration
# ============================================================
GEOJSON_DIR = Path(r"\\kittyserverdw\Andre_kit\data\monkey_fetus\gestational 40\StarDist_12_25_23_monkey\json\geojsons\\32_polys_20x")
MASK_DIR = Path(r"\\kittyserverdw\Andre_kit\data\monkey_fetus\gestational 40\5x\cropped_images\classification_MODEL1_3_28_2024_FINALv2_v2")
BB_DIR = Path(r"\\kittyserverdw\Andre_kit\data\monkey_fetus\gestational 40\2_5x\cropped_images\bounding_boxes")

OUT_DIR = r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\geojson_CODAclass"

# Resolution parameters (MPP)
MPP_20X = 0.4416   # Base resolution (0.4416 um/px)
MPP_BB  = 4        # 2.5x Bounding Box scale
MPP_MASK = 2       # 5x Segmentation Mask scale

LABELS = [
    "bone", "brain", "eye", "heart", "lungs", "GI", "liver", "spleen", 
    "pancreas", "kidney", "mesokidney", "collagen", "ear", "nontissue", 
    "thymus", "thyroid", "bladder", "skull", "spleen2"
]

COLORS = [
    [214, 212, 161], [247, 184, 67], [136, 232, 95], [140, 13, 13], 
    [38, 27, 166], [13, 125, 11], [179, 50, 108], [228, 235, 131], 
    [156, 96, 235], [46, 190, 230], [150, 255, 245], [254, 222, 255], 
    [235, 154, 108], [255, 255, 255], [9, 64, 116], [255, 255, 74], 
    [178, 178, 0], [214, 212, 161], [54, 83, 89]
]

def process_all():
    gj_files = sorted(GEOJSON_DIR.glob("*.geojson"))
    out_path_obj = Path(OUT_DIR)
    out_path_obj.mkdir(parents=True, exist_ok=True)
    
    for gj_path in tqdm(gj_files, desc="Dynamic CODA mapping"):
        mask_path = MASK_DIR / f"{gj_path.stem}.tif"
        if not mask_path.exists():
            mask_path = MASK_DIR / f"{gj_path.stem}.png"
            
        # Bounding box search (recursive, excluding \"old\" folder)
        mat_matches = list(BB_DIR.rglob(f"{gj_path.stem}.mat"))
        mat_matches = [m for m in mat_matches if "old" not in m.parts]
        mat_path = mat_matches[0] if mat_matches else None
        
        if mask_path.exists() and mat_path is not None:
            assign_coda_classifications_to_geojson(
                geojson_path=gj_path,
                mask_path=mask_path,
                mat_path=mat_path,
                out_path=out_path_obj / gj_path.name,
                labels=LABELS,
                colors=COLORS,
                mpp_20x=MPP_20X,
                mpp_bb=MPP_BB,
                mpp_mask=MPP_MASK
            )

if __name__ == "__main__":
    process_all()

Dynamic CODA mapping: 100%|██████████| 24/24 [55:14<00:00, 138.11s/it]


### Visual Integrity Check (First Slide)
Run this cell to verify mapping alignment before/during full processing.

In [ ]:
def _load_bb(mat_path):
    try:
        d = loadmat(str(mat_path))
        arr = d.get("bb") or d[[k for k in d if not k.startswith("__")][0]]
        return arr.flatten()
    except NotImplementedError:
        with h5py.File(str(mat_path), "r") as f:
            key = "bb" if "bb" in f else list(f.keys())[0]
            arr = f[key][()]
        return arr.flatten(order="F") if arr.ndim > 1 else arr.flatten()

def transform_bb_to_target(bb, scale_factor):
    xmin_matlab, xmax_matlab, ymin_matlab, ymax_matlab = bb[0], bb[1], bb[2], bb[3]
    # Convert to 0-indexed
    xmin_0, xmax_0, ymin_0, ymax_0 = xmin_matlab-1, xmax_matlab-1, ymin_matlab-1, ymax_matlab-1
    width_source, height_source = xmax_0 - xmin_0 + 1, ymax_0 - ymin_0 + 1
    
    return xmin_0 * scale_factor, ymin_0 * scale_factor, width_source * scale_factor, height_source * scale_factor

# Scale factors for sanity check verification
SCALE_BB_TO_20X = MPP_BB / MPP_20X
SCALE_20X_TO_5X = MPP_20X / MPP_MASK

# ── Select First Slide ─────────────────────────────────────────────────────────
gj_files = sorted(GEOJSON_DIR.glob("*.geojson"))
if gj_files:
    gj_path = gj_files[0]
    mask_path = MASK_DIR / f"{gj_path.stem}.tif"
    if not mask_path.exists(): mask_path = MASK_DIR / f"{gj_path.stem}.png"
    
    mat_matches = [m for m in BB_DIR.rglob(f"{gj_path.stem}.mat") if "old" not in m.parts]
    mat_path = mat_matches[0] if mat_matches else None

    if mask_path.exists() and mat_path:
        print(f"Viewing First Slide: {gj_path.stem}")
        bb = _load_bb(mat_path)
        xmin_20x, ymin_20x, width_20x, height_20x = transform_bb_to_target(bb, SCALE_BB_TO_20X)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        
        with gj_path.open("r", encoding="utf-8") as f: data = json.load(f)
        feats = data if isinstance(data, list) else data.get("features", [])
        
        # Extract precise polygon centroids (in 20x absolute coordinates)
        cx_list, cy_list = [], []
        for feat in feats:
            ring = feat.get("geometry", {}).get("coordinates", [[]])[0]
            if not ring: continue
            cx, cy = polygon_centroid(ring)
            if cx is not None:
                cx_list.append(cx)
                cy_list.append(cy)
        
        cx_20x, cy_20x = np.array(cx_list), np.array(cy_list)
        
        # Transform centroids: 20x absolute → 5x relative
        cx_5x = (cx_20x - xmin_20x) * SCALE_20X_TO_5X
        cy_5x = (cy_20x - ymin_20x) * SCALE_20X_TO_5X
        
        # ── 3-Panel Visual Integrity Plot ────────────────────────────────────────────
        fig, axes = plt.subplots(1, 3, figsize=(24, 8))
        
        # Panel 1: centroids @ 20x
        axes[0].scatter(cx_20x, cy_20x, s=0.1, alpha=0.5, color='blue')
        rect = plt.Rectangle((xmin_20x, ymin_20x), width_20x, height_20x, fill=False, edgecolor='red', linewidth=1)
        axes[0].add_patch(rect)
        axes[0].set_title(f"1. GeoJSON Centroids @ 20x\n{len(cx_20x):,} nuclei")
        axes[0].invert_yaxis()
        
        # Panel 2: the mask
        axes[1].imshow(mask, cmap="tab20b")
        axes[1].set_title(f"2. CODA 5x Mask\n{mask.shape[1]}x{mask.shape[0]} px")
        
        # Panel 3: overlay
        in_mask = (cx_5x >= 0) & (cx_5x < mask.shape[1]) & (cy_5x >= 0) & (cy_5x < mask.shape[0])
        axes[2].imshow(mask, cmap="tab20b", alpha=0.4)
        axes[2].scatter(cx_5x[in_mask], cy_5x[in_mask], s=0.5, color='lime', label='Within Bounds')
        axes[2].scatter(cx_5x[~in_mask], cy_5x[~in_mask], s=0.5, color='red', label='Out of Bounds', alpha=0.2)
        axes[2].set_title(f"3. Validation Overlay ({in_mask.mean()*100:.1f}% on-target)")
        axes[2].set_xlim(0, mask.shape[1]); axes[2].set_ylim(mask.shape[0], 0)
        axes[2].legend()
        
        plt.tight_layout()
        plt.show()
    else:
        print("Missing required image/data files for visual check.")
else:
    print("No GeoJSON source files found.")